# NB_hospital_daily_summary

Silver → Gold: Hospital Daily Summary. One row per hospital per calendar day with test volumes, doctor/patient counts, abnormal/critical rates, and day-over-day trend.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS

WITH daily_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date AS summary_date,
    COUNT(*) AS total_tests,
    COUNT(DISTINCT lr.patient_id) AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id) AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END) AS critical_test_count
  FROM adb_caresync.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),

with_rates AS (
  SELECT
    da.*,
    CAST(da.abnormal_test_count * 100.0 / da.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(da.critical_test_count * 100.0 / da.total_tests AS DECIMAL(5,2)) AS critical_rate_pct,
    LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date) AS prev_day_total_tests
  FROM daily_agg da
)

SELECT
  wr.hospital_id,
  h.hospital_name,
  wr.summary_date,
  c.day_name,
  c.is_weekend,
  wr.total_tests,
  wr.unique_patients_tested,
  wr.active_doctor_count,
  wr.abnormal_test_count,
  wr.critical_test_count,
  wr.abnormal_rate_pct,
  wr.critical_rate_pct,
  wr.prev_day_total_tests,
  CASE
    WHEN wr.prev_day_total_tests IS NULL OR wr.prev_day_total_tests = 0 THEN NULL
    ELSE CAST((wr.total_tests - wr.prev_day_total_tests) * 100.0 / wr.prev_day_total_tests AS DECIMAL(6,2))
  END AS test_volume_change_pct
FROM with_rates wr
LEFT JOIN adb_caresync.silver.hospitals h
  ON wr.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(wr.summary_date, 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))